In [4]:
pip install seaborn

Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
df = pd.read_csv(
    '/Users/aditikulkarni/Desktop/Supply_Chain_Control_Tower/data/processed/DataCoSupplyChainDataset_Clean.csv',
    parse_dates=[
        'order date (DateOrders)',
        'shipping date (DateOrders)'
    ]
)
df.head()

,Unnamed: 0,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,...,Order State,Order Status,Product Card Id,Product Category Id,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,...,Java Occidental,COMPLETE,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-02-03 22:56:00,Standard Class
1,1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,...,Rajastán,PENDING,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-18 12:27:00,Standard Class
2,2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,...,Rajastán,CLOSED,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-17 12:06:00,Standard Class
3,3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,...,Queensland,COMPLETE,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-16 11:45:00,Standard Class
4,4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,...,Queensland,PENDING_PAYMENT,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-15 11:24:00,Standard Class


#### 1. Order Volumes

In [12]:
# Total unique orders
total_unique_orders = df['Order Id'].nunique()
print("Total unique orders:", total_unique_orders)

Total unique orders: 65752


In [13]:
# Total units sold
total_units_sold = df['Order Item Quantity'].sum()
print("Total units sold:", total_units_sold)

Total units sold: 384079


In [15]:
# Order volume
order_volume = total_units_sold/total_unique_orders
print("Average order volume:", round((order_volume),2))

Average order volume: 5.84


#### 2. Sales and Profitability Analysis:

In [17]:
financial_columns = [
    'Order Id',
    'Order Item Quantity',
    'Order Item Product Price',
    'Sales',
    'Order Item Discount',
    'Order Item Total',
    'Order Profit Per Order',
    'Benefit per order'
]

df[financial_columns].head()

,Order Id,Order Item Quantity,Order Item Product Price,Sales,Order Item Discount,Order Item Total,Order Profit Per Order,Benefit per order
0,77202,1,327.75,327.75,13.110000,314.640015,91.250000,91.250000
1,75939,1,327.75,327.75,16.389999,311.359985,-249.089996,-249.089996
2,75938,1,327.75,327.75,18.030001,309.720001,-247.779999,-247.779999
3,75937,1,327.75,327.75,22.940001,304.809998,22.860001,22.860001
4,75936,1,327.75,327.75,29.500000,298.250000,134.210007,134.210007


In [18]:
# Verify if the sales are actually correct 
calculated_sales = (
    df['Order Item Quantity'] *
    df['Order Item Product Price']
)

sales_difference = (
    df['Sales'] - calculated_sales
).abs()

print("Mismatched records:", (sales_difference > 0.01).sum())
print("Maximum difference:", sales_difference.max())

Mismatched records: 0
Maximum difference: 2.289999997628911e-05


In [27]:
# Total sales
total_sales = df['Sales'].sum()
print(f"Total Sales: ${total_sales:,.2f}")

Total Sales: $36,784,735.01


#### 3. Total Discounts and Net Sales:

In [29]:
# Total discounts
total_discount = df['Order Item Discount'].sum()

print(f"Total Discounts: ${total_discount:,.2f}")

Total Discounts: $3,730,378.40


In [30]:
# Verify net sales 
calculated_net_sales = (
    df['Sales'] - df['Order Item Discount']
)

net_sales_difference = (
    df['Order Item Total'] - calculated_net_sales
).abs()

print("Mismatched records:", 
      (net_sales_difference > 0.01).sum())

print("Maximum difference:", 
      net_sales_difference.max())

Mismatched records: 1224
Maximum difference: 0.010013549999996485


In [32]:
# Total net sales
total_net_sales = df['Order Item Total'].sum()

print(f"Total Net Sales: ${total_net_sales:,.2f}")

Total Net Sales: $33,054,402.38


In [33]:
# Discount percentage
discount_percentage = (total_discount / total_sales) * 100

print(f"Discount Percentage: {discount_percentage:.2f}%")

Discount Percentage: 10.14%


In [34]:
df[
    [
        'Order Id',
        'Order Item Id',
        'Sales',
        'Order Item Total',
        'Benefit per order',
        'Order Profit Per Order'
    ]
].head(10)

,Order Id,Order Item Id,Sales,Order Item Total,Benefit per order,Order Profit Per Order
0,77202,180517,327.75,314.640015,91.250000,91.250000
1,75939,179254,327.75,311.359985,-249.089996,-249.089996
2,75938,179253,327.75,309.720001,-247.779999,-247.779999
3,75937,179252,327.75,304.809998,22.860001,22.860001
4,75936,179251,327.75,298.250000,134.210007,134.210007
5,75935,179250,327.75,294.980011,18.580000,18.580000
6,75934,179249,327.75,288.420013,95.180000,95.180000
7,75933,179248,327.75,285.140015,68.430000,68.430000
8,75932,179247,327.75,278.589996,133.720001,133.720001
9,75931,179246,327.75,275.309998,132.149994,132.149994


In [35]:
profit_difference = (
    df['Benefit per order'] -
    df['Order Profit Per Order']
).abs()

print("Mismatched records:", (profit_difference > 0.01).sum())

print("Maximum difference:", profit_difference.max())

Mismatched records: 0
Maximum difference: 0.0


In [37]:
# Check whether profit is recorded at the order or order-item level
profit_per_order_check = df.groupby('Order Id')[
    'Order Profit Per Order'
].nunique()

print(profit_per_order_check.value_counts().sort_index())

Order Profit Per Order
1    19850
2    11460
3    11400
4    11721
5    11321
Name: count, dtype: int64


In [38]:
calculated_profit = (
    df['Order Item Total'] *
    df['Order Item Profit Ratio']
)

profit_validation_difference = (
    df['Order Profit Per Order'] -
    calculated_profit
).abs()

print("Mismatched records:",
      (profit_validation_difference > 0.01).sum())

print("Maximum difference:",
      profit_validation_difference.max())

Mismatched records: 65390
Maximum difference: 9.299195480020018


The difference is too high, therefore there is some discrepency in the profit analysis.


In [39]:
# Inspect mismatched records:
df.loc[
    profit_validation_difference > 0.01,
    [
        'Order Id',
        'Sales',
        'Order Item Total',
        'Order Item Profit Ratio',
        'Order Profit Per Order',
        'Benefit per order'
    ]
].head(10)

,Order Id,Sales,Order Item Total,Order Item Profit Ratio,Order Profit Per Order,Benefit per order
3,75937,327.75,304.809998,0.08,22.860001,22.860001
5,75935,327.75,294.980011,0.06,18.580000,18.580000
12,75928,327.75,262.200012,0.08,21.760000,21.760000
16,75924,327.75,321.200012,-0.77,-246.360001,-246.360001
17,75923,327.75,317.920013,0.08,23.840000,23.840000
18,75922,327.75,314.640015,0.33,102.260002,102.260002
22,75918,327.75,298.250000,0.08,22.370001,22.370001
24,75916,327.75,288.420013,0.31,90.279999,90.279999
26,75914,327.75,278.589996,0.33,90.540001,90.540001
28,75912,327.75,272.029999,-0.06,-17.139999,-17.139999


In [40]:
# Calculate actual profit ratio
df['Calculated_Profit_Ratio'] = (
    df['Order Profit Per Order'] /
    df['Order Item Total'].replace(0, np.nan)
)

In [41]:
df[
    [
        'Order Item Profit Ratio',
        'Calculated_Profit_Ratio'
    ]
].head(15)

,Order Item Profit Ratio,Calculated_Profit_Ratio
0,0.29,0.290014
1,-0.80,-0.800006
2,-0.80,-0.800013
3,0.08,0.074998
4,0.45,0.449992
5,0.06,0.062987
6,0.33,0.330005
7,0.24,0.239987
8,0.48,0.479989
9,0.48,0.480004


In [47]:
ratio_difference = (
    df['Order Item Profit Ratio'] -
    df['Calculated_Profit_Ratio'].round(2)
).abs()

print("Mismatched ratios:", (ratio_difference > 0.000001).sum())

print("Maximum ratio difference:", ratio_difference.max())

print("Undefined ratios:", ratio_difference.isna().sum())

Mismatched ratios: 7576
Maximum ratio difference: 0.010000043000000014
Undefined ratios: 0


In [48]:
profit_difference = (
    df['Benefit per order'] -
    df['Order Profit Per Order']
).abs()

print("Mismatched records:", (profit_difference > 0.01).sum())

print("Maximum difference:", profit_difference.max())

Mismatched records: 0
Maximum difference: 0.0


#### 4. Total Profit

In [49]:
# Total Profit:
total_profit = df['Order Profit Per Order'].sum()

print(f"Total Profit: ${total_profit:,.2f}")

Total Profit: $3,966,902.97


In [50]:
# Overall Profit Margin:
profit_margin = (total_profit / total_net_sales) * 100

print(f"Profit Margin: {profit_margin:.2f}%")

Profit Margin: 12.00%


In [53]:
# Calculate Average Order Value:
average_order_value = total_net_sales / total_unique_orders

print(f"Average Order Value: ${average_order_value:,.2f}")

Average Order Value: $502.71


#### Loss-Making Order Analysis

In [54]:
order_profit = df.groupby('Order Id')[
    'Order Profit Per Order'
].sum().reset_index()

order_profit.head(10)

,Order Id,Order Profit Per Order
0,1,88.790001
1,2,195.900002
2,4,124.090000
3,5,390.089995
4,7,203.929998
5,8,-39.669994
6,9,119.599999
7,10,93.679999
8,11,273.000000
9,12,406.860010


In [56]:
# Identifying loss making orders
loss_making_orders = order_profit[
    order_profit['Order Profit Per Order'] < 0
]

print("Loss-making orders:", len(loss_making_orders))

Loss-making orders: 13909


In [58]:
# Loss making order percentage
loss_order_percentage = (
    len(loss_making_orders) / total_unique_orders
) * 100

print(f"Loss-making order percentage: {loss_order_percentage:.2f}%")

Loss-making order percentage: 21.15%


In [59]:
# Total financial loss
total_loss = loss_making_orders[
    'Order Profit Per Order'
].sum()

print(f"Total Loss: ${abs(total_loss):,.2f}")

Total Loss: $2,615,203.79


#### 5. Order Status Analysis:

In [60]:
df['Order Status'].unique()

<StringArray>
[       'COMPLETE',         'PENDING',          'CLOSED', 'PENDING_PAYMENT',
        'CANCELED',      'PROCESSING', 'SUSPECTED_FRAUD',         'ON_HOLD',
  'PAYMENT_REVIEW']
Length: 9, dtype: str

In [61]:
df['Order Status'].value_counts()

Order Status
COMPLETE           59491
PENDING_PAYMENT    39832
PROCESSING         21902
PENDING            20227
CLOSED             19616
ON_HOLD             9804
SUSPECTED_FRAUD     4062
CANCELED            3692
PAYMENT_REVIEW      1893
Name: count, dtype: int64

In [62]:
order_status_check = df.groupby('Order Id')[
    'Order Status'
].nunique()

order_status_check.value_counts().sort_index()

Order Status
1    65752
Name: count, dtype: int64

In [63]:
# Find order level status distribution
order_status_counts = df.groupby('Order Status')[
    'Order Id'
].nunique().sort_values(ascending=False)

print(order_status_counts)

Order Status
COMPLETE           21716
PENDING_PAYMENT    14382
PROCESSING          7901
PENDING             7321
CLOSED              7249
ON_HOLD             3624
SUSPECTED_FRAUD     1488
CANCELED            1367
PAYMENT_REVIEW       704
Name: Order Id, dtype: int64


In [65]:
print("Total orders:", order_status_counts.sum())
print("Expected orders:", total_unique_orders)

Total orders: 65752
Expected orders: 65752


In [67]:
order_status_percentage = (
    order_status_counts / total_unique_orders * 100
).round(2)

print(order_status_percentage)

Order Status
COMPLETE           33.03
PENDING_PAYMENT    21.87
PROCESSING         12.02
PENDING            11.13
CLOSED             11.02
ON_HOLD             5.51
SUSPECTED_FRAUD     2.26
CANCELED            2.08
PAYMENT_REVIEW      1.07
Name: Order Id, dtype: float64


In [68]:
df_completed = df[
    df['Order Status'] == 'COMPLETE'
].copy()

print("Completed order items:", len(df_completed))

print("Completed orders:",
      df_completed['Order Id'].nunique())

Completed order items: 59491
Completed orders: 21716


In [69]:
# Calculate completed order net sales
completed_net_sales = df_completed['Order Item Total'].sum()

print(f"Completed Net Sales: ${completed_net_sales:,.2f}")

Completed Net Sales: $10,859,271.73


In [70]:
# Calculate completed order profit
completed_profit = df_completed['Order Profit Per Order'].sum()

print(f"Completed Profit: ${completed_profit:,.2f}")

Completed Profit: $1,321,735.75


In [71]:
# Calculate completed order profit margin
completed_profit_margin = (
    completed_profit / completed_net_sales
) * 100

print(f"Completed Profit Margin: {completed_profit_margin:.2f}%")

Completed Profit Margin: 12.17%


In [72]:
# Calculate completed order AOV
completed_orders = df_completed['Order Id'].nunique()

completed_aov = completed_net_sales / completed_orders

print(f"Completed AOV: ${completed_aov:,.2f}")

Completed AOV: $500.06


#### Financial KPI Summary:

In [74]:
financial_summary = pd.DataFrame({
    'KPI': [
        'Total Orders',
        'Net Sales ($)',
        'Total Profit ($)',
        'Profit Margin (%)',
        'Average Order Value ($)'
    ],
    'All Orders': [
        total_unique_orders,
        total_net_sales,
        total_profit,
        profit_margin,
        average_order_value
    ],
    'Completed Orders': [
        completed_orders,
        completed_net_sales,
        completed_profit,
        completed_profit_margin,
        completed_aov
    ]
})

financial_summary.round(2)

,KPI,All Orders,Completed Orders
0,Total Orders,65752.00,21716.00
1,Net Sales ($),33054402.38,10859271.73
2,Total Profit ($),3966902.97,1321735.75
3,Profit Margin (%),12.00,12.17
4,Average Order Value ($),502.71,500.06


## Financial Analysis — Key Business Insights

- The dataset contains 65,752 unique orders.
- Total recorded net sales amount to `$33.05` million.
- Total discounts amount to `$3.73` million,representing 10.14% of gross sales.
- Total recorded profit is `$3.97` million,with an overall profit margin of 12%.
- The average order value is `$502.71`.
- The average order size is 5.84 units.
- Approximately 21.15% of orders are loss-making, contributing to combined losses of $2.62 million.
- Only 33.03% of orders are marked as COMPLETE.
- Approximately 2.08% of orders are canceled.

## Business Recommendations

- Investigate the drivers of loss-making orders.
- Evaluate the relationship between discounts and profitability.
- Identify underperforming product categories.
- Investigate pending payments and on-hold orders.
- Analyze the relationship between shipping performance and profitability.

## Data Quality and Assumptions

- Financial metrics are calculated at the order-item level and aggregated as required.
- The recorded profit ratio contains rounding or calculation discrepancies.
- Original monetary profit values are retained.
- Duplicate profit columns contain identical values.
- Overall financial KPIs include all order statuses.
- Completed-order KPIs include only COMPLETE orders.
- Recorded sales do not necessarily representrecognized revenue.